In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from scipy.io import loadmat
import mne
from mne.decoding import CSP
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.utils import resample
from sklearn.tree import DecisionTreeClassifier

In [2]:
def load_data(mode='train'):
    path = "../../../../data/20191107/david_20191107_session"
    f_ext = ".npy"
    if mode=='train':
        mode_path = "1_"
        data = pd.DataFrame()
        for i in range(1, 11):
            value = str(i).zfill(2)
            load_path = path + mode_path + value + f_ext
            temp = pd.DataFrame(np.load(load_path))
            data = pd.concat([data, temp])
        return data
    elif mode=='test':
        mode_path = "2_"
        data = pd.DataFrame()
        for i in range(1, 11):
            value = str(i).zfill(2)
            load_path = path + mode_path + value + f_ext
            temp = pd.DataFrame(np.load(load_path))
            data = pd.concat([data, temp])
        return data

In [3]:
def band_pass_filter(eeg, freq_range):
#     13 or 16
    info = mne.create_info(64, 512, ch_types=["eeg"] * 64)
    raw = mne.io.RawArray(eeg.T, info)
    
    iir_params = dict(order=5, ftype='cheby2', rs=2.)
    raw = raw.filter(freq_range[0], freq_range[1], fir_design='firwin', method='iir', iir_params=iir_params)

    return raw._data.T

In [4]:
def drop_classes(df):
    label_not_inc = list(range(2,9))
    indexes_to_drop = []
    i = 0
    while i < len(df):
        if df[66].values[i] in label_not_inc:
            list2 = list(range(i, 767+i))
            i += 767
            indexes_to_drop.extend(list2)
        else:
            i += 1
    indexes_to_keep = set(range(df.shape[0])) - set(indexes_to_drop)
    df_sliced = df.take(list(indexes_to_keep))

    df_sliced = df_sliced.reset_index(drop=True)
    return df_sliced

In [5]:
def data_win(sfreq, data, asynch_label):
    sampling_window = 2 * sfreq
    shift_length = 1 * sfreq
    t_start = 0

    new_data = []
    labels = []

    while t_start + sampling_window < data.shape[0]:
        new_data.append(data[t_start:t_start+sampling_window, :].T)
        labels.append(asynch_label[t_start:t_start+sampling_window])

        t_start = t_start + shift_length

    return np.array(new_data), np.array(labels)

def transform_label(label_new):
    label = []
    for i in label_new:
        count1 = np.count_nonzero(i==1)
        count9 = np.count_nonzero(i==9)
        if count1 >= 512:
            to_add = 1
        elif count9 >= 512:
            to_add = 2
        else:
            to_add = 0
        label.append(to_add)
        
    label = np.array(label)
    return label

In [6]:
def prune_records(data_new, label):
    train_data = []
    label_data = []

    for i in range(len(label)):
        if label[i] != 0:
            label_data.append(label[i])
            train_data.append(data_new[i])
    label_data = np.array(label_data)
    train_data = np.array(train_data)

    return train_data, label_data

In [7]:
def fbcsp(df, labels, sfreq, train=True, csp_objects=None):
    freq = 4
    increment = 4
    end_freq = 40
    if train==True:
        csp_objects = []
        csp_data = []
        while freq < end_freq:
            freq_range = []
            freq_range.append(freq)
            freq_range.append(freq+increment)
            freq += increment
            out_data = band_pass_filter(df.iloc[:, :-1].values, freq_range=freq_range)

            X, y = data_win(sfreq=sfreq, data=out_data, asynch_label=labels)
            y = transform_label(y)
            X, y = prune_records(X, y)

            csp = CSP(n_components=2, reg=None, log=True, norm_trace=False)
            final_data = csp.fit_transform(X, y)

            csp_objects.append(csp)
            csp_data.append(final_data)

        return np.array(csp_objects), np.array(csp_data), y

    else:
        csp_data = []
        count = 0
        while freq < end_freq:
            freq_range = []
            freq_range.append(freq)
            freq_range.append(freq+increment)
            freq += increment
            out_data = band_pass_filter(df.iloc[:, :-1].values, freq_range=freq_range)

            X, y = data_win(sfreq=sfreq, data=out_data, asynch_label=labels)
            y = transform_label(y)
            X, y = prune_records(X, y)

            final_data = csp_objects[count].transform(X)
            count += 1

            csp_data.append(final_data)

        return np.array(csp_data), y


In [8]:
def itr(n_class, p_class, c_time):
    B = (np.log2(n_class) + (p_class * np.log2(p_class)) + ((1-p_class) * np.log2((1-p_class)/(n_class-1)))) / c_time * 60

    return B

def performance_metrics(y_test, y_pred):
    acc = accuracy_score(y_test, y_pred)
    print('Accuracy Score: ', acc)
    print('Cohen Kappa Score: ', cohen_kappa_score(y_test, y_pred))
    print('ITR (bits per minute): ', itr(n_class=2, p_class=acc, c_time=2))
    print('Confusion Matrix: ', confusion_matrix(y_test, y_pred))

In [9]:
def get_train_data():
    sfreq = 512
    trigger_points = {1:4, 2:4}
    
    df = load_data(mode='train')
    train_df = drop_classes(df)
    
    train_df = train_df.drop([64, 65], axis=1)
    csp_objects, csp_data, y = fbcsp(train_df, train_df.iloc[:, -1].values, sfreq, train=True)
    
    final_data = pd.DataFrame(csp_data[0])
    col_count = 4
    for i in range(1, len(csp_data)):
        for j in range(len(csp_data[i].T)):
            final_data[str(col_count)] = csp_data[i].T[j]
            col_count += 1

    return final_data.values, y, csp_objects

def get_eval_data(csp_objects):
    sfreq = 512
    trigger_points = {1:4, 2:4}
    
    df = load_data(mode='test')
    train_df = drop_classes(df)
    train_df = train_df.drop([64, 65], axis=1)
    
    csp_data, y = fbcsp(train_df, train_df.iloc[:, -1].values, sfreq, train=False, csp_objects=csp_objects)
    
    final_data = pd.DataFrame(csp_data[0])
    col_count = 4
    for i in range(1, len(csp_data)):
        for j in range(len(csp_data[i].T)):
            final_data[str(col_count)] = csp_data[i].T[j]
            col_count += 1

    return final_data.values, y

In [10]:
def train_mibif(X, y):
    #### resample and stratify####
    data = pd.DataFrame(X)
    data['class'] = y
    
    df_majority = data[data['class']==1]
    df_minority = data[data['class']==2]
    
    # Upsample minority class
    df_minority_upsampled = resample(df_minority, 
                                 replace=True,     # sample with replacement
                                 n_samples=118,    # to match majority class
                                 random_state=123) # reproducible results
    
    
    # Combine majority class with upsampled minority class
    df_upsampled = pd.concat([df_majority, df_minority_upsampled])
    
    X = df_upsampled.iloc[:, :-1].values
    y = df_upsampled.iloc[:, -1].values
    
    print('------Train---------')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

  #get the best k features base on MIBIF algorithm
    select_K = SelectKBest(mutual_info_classif,k=10).fit(X, y)
    extra = select_K.get_support()
    if extra[0] == True:
        extra[1] = True
    for i in range(2, len(extra)):
        if extra[i] == True:
            if i%2 == 0:
                extra[i+1] = True
            else:
                extra[i-1] = True
    pos = np.where(extra==False)
    New_train = np.delete(X_train, list(pos[0]), 1)
    New_test = np.delete(X_test, list(pos[0]), 1)
    # New_train=select_K.transform(X_train)
    # New_test=select_K.transform(X_test)
    ss = StandardScaler()
    New_train = ss.fit_transform(New_train,y_train)
    New_test = ss.transform(New_test)

    print('####### SVM#####')
    svm = SVC()
    svm.fit(New_train, y_train)
    y_pred = svm.predict(New_test)
    performance_metrics(y_test, y_pred)

    print('##########LDA#########')
    lda = LinearDiscriminantAnalysis()
    lda.fit(New_train, y_train)
    y_pred = lda.predict(New_test)
    performance_metrics(y_test, y_pred)

    return list(pos[0]), svm, lda, ss

def eval_mibif(pos, svm, lda, ss, X, y):
    print('------Test--------')
    X = np.delete(X, pos, 1)
    X = ss.transform(X)
    print('#####SVM######')
    y_pred = svm.predict(X)
    performance_metrics(y, y_pred)
    print('#####LDA######')
    y_pred = lda.predict(X)
    performance_metrics(y, y_pred)

In [11]:
def train_rf(X, y):
    #### resample and stratify####
    data = pd.DataFrame(X)
    data['class'] = y
    
    df_majority = data[data['class']==1]
    df_minority = data[data['class']==2]
    
    # Upsample minority class
    df_minority_upsampled = resample(df_minority, 
                                 replace=True,     # sample with replacement
                                 n_samples=118,    # to match majority class
                                 random_state=123) # reproducible results
    
    
    # Combine majority class with upsampled minority class
    df_upsampled = pd.concat([df_majority, df_minority_upsampled])

#     df_majority_downsampled = resample(df_majority, 
#                                  replace=True,     # sample with replacement
#                                  n_samples=66,    # to match majority class
#                                  random_state=123) # reproducible results
    
    
#     # Combine majority class with upsampled minority class
#     df_upsampled = pd.concat([df_minority, df_majority_downsampled])
    
    X = df_upsampled.iloc[:, :-1].values
    y = df_upsampled.iloc[:, -1].values
    print('--------Train--------')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
    clf = RandomForestClassifier(random_state=42)
    ss = StandardScaler()
    X_train = ss.fit_transform(X_train, y_train)
    clf.fit(X_train, y_train)
    X_test = ss.transform(X_test)
    y_pred = clf.predict(X_test)
    ####Random Forest#######
    performance_metrics(y_test, y_pred)

    return clf, ss
def eval_rf(clf, ss, X, y):
    print('--------Test------')
    X = ss.transform(X)
    y_pred = clf.predict(X)
    performance_metrics(y, y_pred)

In [12]:
X, y, csp_objects = get_train_data()
X_eval, y_eval = get_eval_data(csp_objects=csp_objects)

Creating RawArray with float64 data, n_channels=64, n_times=233596
    Range : 0 ... 233595 =      0.000 ...   456.240 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Computing data rank from raw with rank=None
    Using tolerance 2.8e+02 (2.2e-16 eps * 64 dim * 2e+16  max singular value)
    Estimated rank (mag): 64
    MAG: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating covariance using EMPIRICAL
Done.
Computing data rank from raw with rank=None
    Using tolerance 2e+02 (2.2e-16 eps * 64 dim * 1.4e+16  max singular value)
    Estimated rank (mag): 64
    MAG: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating 

Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=64, n_times=233596
    Range : 0 ... 233595 =      0.000 ...   456.240 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 36 - 40 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 36.00, 40.00 Hz: -4.00, -4.00 dB

Computing data rank from raw with rank=None
    Using tolerance 0.23 (2.2e-16 eps * 64 dim * 1.6e+13  max singular value)
    Estimated rank (mag): 64
    MAG: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating covariance using EMPIRICAL
Done.
Computing data rank from raw with rank=None
    Using tolerance 0.16 (2.2e-16 eps * 64 dim * 1.1e+13  max singular value)
    Estimated rank (mag): 64
    MAG: rank 64 computed from 64 data channels with 0 projector

In [13]:
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)

------Train---------
####### SVM#####
Accuracy Score:  0.9322033898305084
Cohen Kappa Score:  0.8645235361653272
ITR (bits per minute):  19.270598812631
Confusion Matrix:  [[27  3]
 [ 1 28]]
##########LDA#########
Accuracy Score:  0.8813559322033898
Cohen Kappa Score:  0.7627800114876508
ITR (bits per minute):  14.236479602741381
Confusion Matrix:  [[26  4]
 [ 3 26]]
------Test--------
#####SVM######
Accuracy Score:  0.7386363636363636
Cohen Kappa Score:  0.0
ITR (bits per minute):  5.136085362299553
Confusion Matrix:  [[130   0]
 [ 46   0]]
#####LDA######
Accuracy Score:  0.7443181818181818
Cohen Kappa Score:  0.03178484107579471
ITR (bits per minute):  5.395199600467673
Confusion Matrix:  [[130   0]
 [ 45   1]]


In [14]:
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

--------Train--------
Accuracy Score:  0.9830508474576272
Cohen Kappa Score:  0.9661114302125215
ITR (bits per minute):  26.281503968669778
Confusion Matrix:  [[29  1]
 [ 0 29]]
--------Test------
Accuracy Score:  0.7386363636363636
Cohen Kappa Score:  0.0
ITR (bits per minute):  5.136085362299553
Confusion Matrix:  [[130   0]
 [ 46   0]]
